# Credit Card Fraud Detection
### XGBoost + SMOTE on Heavily Imbalanced Transaction Data

Detects fraudulent transactions in a heavily imbalanced dataset (fraud typically
<1% of transactions, as in Kaggle's IEEE-CIS / `creditcard.csv` datasets).

**Approach:** XGBoost (captures non-linear interactions between amount, time,
device, velocity) + SMOTE oversampling applied **only to the training fold**,
threshold tuning against precision/recall and cost trade-offs, and feature
importance for interpretation.

This notebook uses a **synthetic** dataset generator so it runs out-of-the-box.
To use real data, replace `generate_synthetic_data()` with a `pd.read_csv(...)` call
on the Kaggle "Credit Card Fraud Detection" (V1-V28 PCA features) or "IEEE-CIS Fraud
Detection" dataset, and update `FEATURE_COLUMNS` accordingly.

> **Requires:** `xgboost` and `imbalanced-learn` (`pip install xgboost imbalanced-learn`)


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, confusion_matrix, classification_report, f1_score
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

RANDOM_STATE = 42


## 2. Data

Synthetic transaction data mimicking a heavily imbalanced fraud dataset (~0.6% fraud
rate). Fraudulent transactions are modeled with higher amounts, a skew toward
late-night hours, greater distance from home, higher transaction velocity, and a
higher chance of a new device.

In [ ]:
def generate_synthetic_data(n_samples=50_000, fraud_rate=0.006, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n_fraud = int(n_samples * fraud_rate)
    n_legit = n_samples - n_fraud

    def _night_weighted_hours():
        w = np.ones(24)
        w[0:5] *= 3  # fraud skews toward late-night hours
        return w / w.sum()

    def make_block(n, fraud_flag):
        if fraud_flag:
            amount = rng.gamma(3, 150, n)
            hour = rng.choice(np.arange(24), n, p=_night_weighted_hours())
            distance_from_home = rng.exponential(80, n)
            num_txn_last_hour = rng.poisson(4, n)
            is_new_device = rng.binomial(1, 0.6, n)
            merchant_risk_score = rng.beta(4, 2, n)
        else:
            amount = rng.gamma(2, 40, n)
            hour = rng.integers(0, 24, n)
            distance_from_home = rng.exponential(10, n)
            num_txn_last_hour = rng.poisson(0.5, n)
            is_new_device = rng.binomial(1, 0.05, n)
            merchant_risk_score = rng.beta(2, 5, n)

        return pd.DataFrame({
            "amount": amount,
            "hour": hour,
            "distance_from_home_km": distance_from_home,
            "num_txn_last_hour": num_txn_last_hour,
            "is_new_device": is_new_device,
            "merchant_risk_score": merchant_risk_score,
            "is_fraud": fraud_flag,
        })

    df = pd.concat([make_block(n_legit, 0), make_block(n_fraud, 1)], ignore_index=True)
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df


FEATURE_COLUMNS = [
    "amount", "hour", "distance_from_home_km",
    "num_txn_last_hour", "is_new_device", "merchant_risk_score",
]
TARGET_COLUMN = "is_fraud"

df = generate_synthetic_data()
print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df[TARGET_COLUMN].mean():.3%}")
df.head()


## 3. Train / Test Split

Stratified split so both sets preserve the (tiny) fraud rate.

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train fraud rate: {y_train.mean():.3%}, Test fraud rate: {y_test.mean():.3%}")


## 4. Scale + SMOTE (Training Fold Only!) + Train XGBoost

**Critical:** SMOTE is applied only to the training data, *after* the train/test split.
Applying it before the split would let synthetic points derived from test-set
neighbors leak into training (or vice versa), inflating validation metrics.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_train_scaled, y_train)
print(f"Before SMOTE: {np.bincount(y_train)}")
print(f"After  SMOTE: {np.bincount(y_res)}")

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_res, y_res)


## 5. Evaluate on the Untouched, Imbalanced Test Set

This reflects real-world traffic, which is imbalanced too.

In [ ]:
X_test_scaled = scaler.transform(X_test)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
print(f"Test ROC-AUC: {roc_auc:.4f}")
print(f"Test PR-AUC : {pr_auc:.4f}  (more informative given ~{y_test.mean():.2%} fraud rate)")

y_pred_default = (y_proba >= 0.5).astype(int)
print("\n--- Classification report @ threshold = 0.5 ---")
print(classification_report(y_test, y_pred_default, digits=3))


## 6. Threshold Tuning

Two strategies:
1. **F1-optimal** — maximizes the harmonic mean of precision and recall; a reasonable
   default with no explicit cost data.
2. **Cost-based** — minimizes `FN x cost_FN + FP x cost_FP` given business cost
   estimates (missed fraud vs. false-alarm friction).

In [ ]:
def tune_threshold_by_f1(y_test, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx = np.argmax(f1_scores[:-1])
    best_threshold = thresholds[best_idx]

    print(f"Threshold = {best_threshold:.3f}  |  F1 = {f1_scores[best_idx]:.3f}  "
          f"|  Precision = {precisions[best_idx]:.3f}  |  Recall = {recalls[best_idx]:.3f}")

    y_pred = (y_proba >= best_threshold).astype(int)
    print("\n--- Classification report @ F1-optimal threshold ---")
    print(classification_report(y_test, y_pred, digits=3))
    return best_threshold

f1_threshold = tune_threshold_by_f1(y_test, y_proba)


In [ ]:
def cost_based_threshold(y_test, y_proba, cost_fn=50, cost_fp=1):
    thresholds = np.linspace(0.01, 0.99, 99)
    costs = []
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        costs.append(fn * cost_fn + fp * cost_fp)

    best_idx = int(np.argmin(costs))
    best_threshold = thresholds[best_idx]
    print(f"Cost ratio FN:FP = {cost_fn}:{cost_fp}")
    print(f"Optimal threshold = {best_threshold:.2f}  |  Min expected cost = {costs[best_idx]:.0f}")

    y_pred_opt = (y_proba >= best_threshold).astype(int)
    print("\n--- Classification report @ cost-optimal threshold ---")
    print(classification_report(y_test, y_pred_opt, digits=3))
    return best_threshold

cost_threshold = cost_based_threshold(y_test, y_proba, cost_fn=50, cost_fp=1)


## 7. Feature Importance (XGBoost gain)

In [ ]:
importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "gain_importance": model.feature_importances_,
}).sort_values("gain_importance", ascending=False)

print(importance_df.to_string(index=False))


## 8. Plots — ROC Curve, PR Curve, Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, y_proba):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

precisions, recalls, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(recalls, precisions,
             label=f"PR-AUC = {average_precision_score(y_test, y_proba):.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve (key metric for imbalanced fraud)")
axes[1].legend()

axes[2].barh(importance_df["feature"][::-1], importance_df["gain_importance"][::-1])
axes[2].set_xlabel("Gain importance")
axes[2].set_title("XGBoost Feature Importance")

plt.tight_layout()
plt.show()


## 9. Summary

- **Model:** XGBoost, tuned for tabular imbalanced data (shallow-ish trees,
  subsampling, `eval_metric='aucpr'`).
- **Imbalance handling:** SMOTE applied to the training fold only, after the split
  (avoids data leakage). The untouched, imbalanced test set is used for evaluation,
  matching real-world deployment conditions.
- **Metrics:** PR-AUC emphasized over ROC-AUC, since ROC-AUC can look artificially
  strong under extreme imbalance (~0.6% fraud rate here).
- **Threshold:** tuned two ways — F1-optimal (no cost data needed) and cost-based
  (given FN:FP cost ratio, default 50:1 here — tune to your actual chargeback and
  review costs).
- **Interpretability:** gain-based feature importance highlights transaction velocity,
  new-device flag, and merchant risk score as top fraud signals.

**Next steps:** try `scale_pos_weight` in XGBoost as an alternative to SMOTE, use
SHAP for per-transaction explanations, switch to a time-based train/test split to
account for fraud pattern drift, and frame results as precision@k / alerts-per-day
for a fraud ops team.
